# open the tiff data

to get familiarized with it


In [ ]:
from pathlib import Path
import json
import xarray as xr

## virtualizarr specific libraries
from virtualizarr import open_virtual_dataset
from virtual_tiff import VirtualTIFF
from obstore.store import LocalStore
from obspec_utils.registry import ObjectStoreRegistry

# pydap specific libraries
from pydap.model import DatasetType
from pydap.responses.dmr import DMRResponse
from pydap.parsers.dmr import DummyData

## Kerchunk files

## Input tif files

We work directly with the tif file.



In [ ]:
directory_path = Path("./input_files/")

# Find all .json files in the top-level directory
tif_files = list(directory_path.glob("*.tif"))
print("found: ", len(tif_files), " tif files")
filename = tif_files[0]
filename

## Virtual datastore

The end result is an Xarray Dataset object. Will use an intermediate product, to translate the information into the pydap data model


In [ ]:
filepath = f"{filename.resolve().parent}/{filename.name}"

registry = ObjectStoreRegistry({"file://": LocalStore()})
parser = VirtualTIFF(ifd_layout="nested")

ms = parser(f"file://{filepath}", registry=registry)
ds = ms.to_virtual_datatree()
ds

## Parsing metadata

The following steps will enable the creation of a pydap dataset, and with it, the generation of a dmrpp



In [ ]:
groups = list(ms._group.groups.keys())
groups

In [ ]:
ms._group.arrays # root arrays?

In [ ]:
ms._group.metadata.attributes

## Hierarchical data

Extract any nested data and place it within Groups


In [ ]:
GROUPS = {}
for group in groups:
    arrays={}
    for k,v in ms._group[group].arrays.items():
        chunk_manifest = {"chunk_shape": v.metadata.chunk_grid.chunk_shape, "fill_value": v.metadata.fill_value, "codecs": v.metadata.codecs,"hrefs": v.manifest.dict()}
        arrays.update({k:{'shape': v.shape, 'dtype': v.dtype, "dims": v.metadata.dimension_names, "chunk_manifest": chunk_manifest}})
    GROUPS.update({group:{"attributes": ms._group[group].metadata.attributes, "arrays": arrays}})

The nested dictionary holds information about the array data that will go into the dmrpp

In [ ]:
GROUPS['0']['arrays']['0'].keys()

In [ ]:
GROUPS['0']['arrays']['0']['chunk_manifest'].keys()

In [ ]:
GROUPS['0']['arrays']['0']['chunk_manifest']['codecs']

In [ ]:
ms._group['0'].arrays['0']

In [ ]:
ms._group['0'].arrays['0'].metadata.codecs

## Populate the pydap dataset

Here, the virtualizarr metadata is injected into the pydap data model


In [ ]:
def array_metadata(arraymeta: dict, parent: str | None = None):
    """Reads array medatata extracteed from virtualizarr, and
    and re structures it following pydap-specific syntax
    """
    if not parent:
        parent = "/"
    _dims_shapes = dict((dim, size) for dim, size in zip(arraymeta['dims'], arraymeta['shape']))
    _dims = ["/".join([parent,dim]) for dim in list(_dims_shapes)]
    _data = DummyData(dtype= arraymeta['dtype'], shape= arraymeta['shape'], path = parent)
    return _dims, _dims_shapes, _data

In [ ]:
pyds = DatasetType(name=filename.name, attributes=ms._group.metadata.attributes)
for array in ms._group.arrays:
    dims = ms._group[array].metadata.dimensions_names
    data = DummyData(dtype=ms._group[array].dtype, shape=ms._group[array].shape, path='/')
    pyds.createVariable(name=array, dims=dims, data=data)

### Now all hierarchical data
for gr in GROUPS:
    _DIMS = {}
    group_name = "/" + gr
    pyds.createGroup(name=group_name, attributes=GROUPS[gr]['attributes'])
    for array in GROUPS[gr]["arrays"]:
        var_name = "/".join([group_name,array])
        dims, dim_shapes, data = array_metadata(arraymeta = GROUPS[gr]["arrays"][array], parent=group_name)
        pyds.createVariable(name = var_name, dims = dims, data=data)
        _DIMS.update(dim_shapes)
    pyds[group_name].attributes['dimensions'] = _DIMS

In [ ]:
dmr_name = f"./output_files/{filename.name}.dmr"
dmr_name

In [ ]:
with open(dmr_name, "wb") as file:
    file.write(b"".join(DMRResponse(pyds)).decode("ascii").encode("utf-8"))

# What is missing?

All the `++` elements of the dmrpp. All the information has been extracted in the `GROUPS` nested dictionary. For example:

```python
gr = "<group_name_here>"
array = "<array_name_here>"
GROUPS[gr]["arrays"][array]["chunk_manifest"].keys()
>>> dict_keys(['chunk_shape', 'fill_value', 'codecs', 'hrefs'])
```



## What is needed

1. Translate the zarr chunk syntax into the dmrpp chunk synthax. For example in the case the `chunk_shape = (512,512)`:

* Chunks in zarr -> {'0.0', '0.1', ...}

* chunks in dmrpp -> {'[512, 512]', '[512, 1024]'}

* 0.0 -> [256,256]

* 0.1 -> [512, 1024]

2. Once that is translated, this informations needs to be added into the pydap dataset, per array. The easiest way is to expand on the `DummyData` class defined in `pydap.parsers.dmr`. We need to add all the (basic) elements of the dmrpp into this class. Currently the DummyData class only takes `dtype`, `shape`, and `path` (which is the parent). Every new element must be optional so it does not break any existing workflow.

3. Once the `DummyClass` has been expanded to include all (optional) dmrpp elements, the `DMRResponse` in `pydap.responses.dmr` needs to be expanded to extract this (optional) information per array. This is should not be too hard, since almost all machinery is there already.




In [ ]:
GROUPS['0']['arrays']['0']['chunk_manifest'].keys()

In [ ]:
GROUPS['0']['arrays']['0']['chunk_manifest']